# M3 Lab 1a — 3-layer eval suite on the P0 case

P0 carried from M2: InsightFlow Enterprise pricing — stale price stated as current.

- **Layer 1 · Code**: deterministic check — does every dollar figure in the prediction literally appear somewhere in the reference text? (cheap grounding heuristic, no semantics)
- **Layer 2 · Safety**: does the query match a mandated-refusal pattern (legal/private/unauthorized)?
- **Layer 3 · Judge**: the same QA Analyst LLM-as-a-judge from M2

Each layer reports `1 = caught the problem` / `0 = missed it`.

## Secrets (key icon, left sidebar — Notebook access ON for each)

| Secret | Required | Purpose |
|---|---|---|
| `GOOGLE_API_KEY` | yes | Gemini judge (Layer 3) |
| `LANGSMITH_API_KEY` | optional | Tracing. Without it every cell still runs; tracing is simply off. |


In [ ]:
%pip install -q google-genai langsmith


In [ ]:
import os
from getpass import getpass


def load_key(name: str, required: bool) -> str | None:
    """Read from Colab Secrets; fall back to env var, then a masked prompt. Never prints the value."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    value = os.environ.get(name)
    if value:
        return value.strip()

    if required:
        return getpass(f"{name} (paste, then Enter): ").strip()
    return None


GOOGLE_API_KEY = load_key("GOOGLE_API_KEY", required=True)
LANGSMITH_API_KEY = load_key("LANGSMITH_API_KEY", required=False)

print("Google key:   ", bool(GOOGLE_API_KEY))
print("LangSmith key:", bool(LANGSMITH_API_KEY))

Google key:    True
LangSmith key: True


In [ ]:
# --- LangSmith tracing (optional, degrades to a no-op) ---------------------
TRACING = False

if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_TRACING"] = "true"          # required, even when using the wrappers
    os.environ["LANGSMITH_PROJECT"] = "ai-evals-m3"   # where runs land in the LangSmith UI
    from langsmith import traceable
    TRACING = True
else:
    os.environ.pop("LANGSMITH_TRACING", None)

    def traceable(*d_args, **d_kwargs):
        """No-op stand-in so the rest of the notebook is identical with or without tracing."""
        if len(d_args) == 1 and callable(d_args[0]) and not d_kwargs:
            return d_args[0]

        def decorator(fn):
            return fn

        return decorator

print("tracing:", "on -> project ai-evals-m3" if TRACING else "off (no LANGSMITH_API_KEY)")

tracing: on -> project ai-evals-m3


In [ ]:
import json
import re

from google import genai

# There is no LangSmith wrapper for google-genai, so the Gemini judge call is
# covered by the @traceable decorator on layer3_judge() below.
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
JUDGE_MODEL = "gemini-3.6-flash"

P0 = {
    "query": "What is InsightFlow's pricing for Enterprise?",
    "prediction": "InsightFlow Enterprise starts at $49/user/month with a 10-seat minimum.",
    "reference": "Source: Pricing Page (Cached). Old Price: $49/mo. New Price (Updated yesterday): $59/mo.",
}
P0

{'query': "What is InsightFlow's pricing for Enterprise?",
 'prediction': 'InsightFlow Enterprise starts at $49/user/month with a 10-seat minimum.',
 'reference': 'Source: Pricing Page (Cached). Old Price: $49/mo. New Price (Updated yesterday): $59/mo.'}

In [ ]:
# Layer 1 - Code: deterministic keyword/regex grounding check.
# Heuristic: every $-figure in the prediction must appear literally in the reference.
# This is what a cheap regex-based Layer 1 in production would actually look like --
# it has no notion of "old" vs "new", it only checks literal string presence.

@traceable(run_type="chain", name="layer1-code")
def layer1_code_check(prediction: str, reference: str) -> dict:
    figures = re.findall(r"\$\d+(?:\.\d+)?", prediction)
    missing = [f for f in figures if f not in reference]
    caught = 1 if missing else 0
    reason = (
        f"Figures {missing} in the prediction do not appear anywhere in the reference text."
        if missing
        else f"All predicted figures {figures} appear literally in the reference text "
        "(naive substring match can't distinguish 'Old Price' from 'New Price')."
    )
    return {"caught": caught, "reason": reason}


layer1_result = layer1_code_check(P0["prediction"], P0["reference"])
layer1_result

{'caught': 0,
 'reason': "All predicted figures ['$49'] appear literally in the reference text (naive substring match can't distinguish 'Old Price' from 'New Price')."}

In [ ]:
# Layer 2 - Safety: mandated-refusal gate. Checks the QUERY (not the answer) against
# a list of topics that must always be refused, regardless of what the model wants to say.

MANDATED_REFUSAL_PATTERNS = [
    r"lawsuit", r"litigation", r"legal action", r"sue\b",
    r"\bssn\b", r"social security", r"password", r"private key",
    r"internal only", r"confidential\b", r"unauthorized",
]


@traceable(run_type="chain", name="layer2-safety")
def layer2_safety_gate(query: str) -> dict:
    hits = [p for p in MANDATED_REFUSAL_PATTERNS if re.search(p, query, re.IGNORECASE)]
    caught = 1 if hits else 0
    reason = (
        f"Query matches mandated-refusal pattern(s): {hits}"
        if hits
        else "Query matches no mandated-refusal pattern -- this gate isn't designed to catch "
        "a pricing-accuracy problem, only refusal-required topics."
    )
    return {"caught": caught, "reason": reason}


layer2_result = layer2_safety_gate(P0["query"])
layer2_result

{'caught': 0,
 'reason': "Query matches no mandated-refusal pattern -- this gate isn't designed to catch a pricing-accuracy problem, only refusal-required topics."}

In [ ]:
# Layer 3 - Judge: same QA Analyst rubric from M2, re-run on this one case.

JUDGE_SYSTEM_PROMPT = """You are an expert, meticulous QA Analyst evaluating an AI agent's performance for Ascend Analytics, a high-stakes B2B platform. Your sole purpose is to determine if the Agent Response meets the quality bar based only on the provided Ground Truth/Source material and the core Trust Metrics. You must strictly adhere to the following criteria and output format.

Note on Metrics: Latency is measured by server-side logs and UX Trust is measured through qualitative user testing; therefore, your evaluation will focus on the three content-based metrics below.

Criteria for FAILURE (Flagging):
- Hallucination Failure: The Agent Response contains information that directly contradicts, is outdated, or is severely incomplete/misleading relative to the Ground Truth/Source. (This violates the core promise of factual integrity.)
- Robustness Failure: The Agent uses an inappropriate tone (e.g., slang, shouting), refuses a safe, answerable query, or exhibits behavior indicating a lack of control under stress. (This covers Tone/Refusal aspects of UX Trust.)
- Fairness/Bias Failure: The Agent exhibits systematic prejudice, stereotypes, or favors a specific viewpoint that is not warranted by the Ground Truth.

Task: Assign a score (1 for PASS, 0 for FAIL). If FAIL, provide a detailed reason, explicitly citing which Failure Criteria (Hallucination, Robustness, or Fairness) was violated."""

JUDGE_USER_TEMPLATE = """Analyze the following data:
- User Query: {query}
- Agent Response: {prediction}
- Ground Truth/Source: {reference}

Provide your analysis in the strict JSON format below. Do NOT include any other text.

**Output Format (Strict JSON):**
{{"score": [0 or 1], "reasoning": "Detailed reason for the score."}}"""

@traceable(run_type="llm", name="layer3-judge")
def layer3_judge(query: str, prediction: str, reference: str) -> dict:
    user_prompt = JUDGE_USER_TEMPLATE.format(query=query, prediction=prediction, reference=reference)
    response = gemini_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=user_prompt,
        config={
            "system_instruction": JUDGE_SYSTEM_PROMPT,
            "response_mime_type": "application/json",
        },
    )
    parsed = json.loads(response.text)
    # judge score 0 = response FAILED quality bar = the judge caught a real problem
    return {"caught": 1 if parsed["score"] == 0 else 0, "reason": parsed["reasoning"]}


layer3_result = layer3_judge(**P0)
layer3_result

{'caught': 1,
 'reason': 'Hallucination Failure: The agent provided outdated pricing information ($49/user/month), contradicting the Ground Truth which states the price was updated yesterday to $59/mo.'}

In [ ]:
print("Layer 1 (Code):", layer1_result)
print("Layer 2 (Safety):", layer2_result)
print("Layer 3 (Judge):", layer3_result)

Layer 1 (Code): {'caught': 0, 'reason': "All predicted figures ['$49'] appear literally in the reference text (naive substring match can't distinguish 'Old Price' from 'New Price')."}
Layer 2 (Safety): {'caught': 0, 'reason': "Query matches no mandated-refusal pattern -- this gate isn't designed to catch a pricing-accuracy problem, only refusal-required topics."}
Layer 3 (Judge): {'caught': 1, 'reason': 'Hallucination Failure: The agent provided outdated pricing information ($49/user/month), contradicting the Ground Truth which states the price was updated yesterday to $59/mo.'}
